# just_drop vs just_drop_hal：处理 PPS

**比什么。** 同一块 ESP32-C3、同一 AP、同一 TR3000 发包机上，比较两条数据面：HAL（`just_drop_hal`，应用层 UDP:8080 主动丢弃并计 `drop`）与 XDP（`just_drop` 固件 + 冷表 `just_drop.elf`，Wi-Fi 入口 XDP_DROP 计 `drop`）。指标是进入各自丢弃路径的处理 PPS，不是无线物理层接收 PPS；驱动在计数点之前丢掉的帧看不见。

**怎么跑。** Mac 只开 `cargo xtask agent --listen 0.0.0.0:9240 --no-monitor`（cwd=`xpost/`）；收到烧录请求时按 `profiles.toml` 的 `build` 编译再烧。TR3000 上 `./xpost-test`（编译期嵌入 `config/test.toml`：默认 3 对、每臂 40 s、1 s 采样、`formal=true`）。每一对先烧 HAL 再烧 XDP：agent `FlashProfile` → 等启动 → `ping` → HAL 再发 `drop on` → 开测。库写入 schema `formal` 的 `experiment_*`，开跑前清空表。本笔记本只读库做性能汇总与图，不烧录、不发包。

**包怎么发。** TR3000 用 AF_PACKET RAW 向 DUT 的 `esp_ip:8080` 打满速 UDP（固定小包、多线程 `udp_blast`），源 IP 伪造为同网段地址；窗口默认 40 s，主机侧记录成功提交发送数 → `host_pps`。DUT 不解析 payload；串口只读单调 `drop`（及对照用的 `net_rx`）。XDP 臂要求窗口内 `net_rx` 不增（包未进协议栈）；任一臂 `drop` 不增则失败。

**数怎么记、怎么算。** 发包开始前读一次 `counters` 作起点，窗口内约每 1 s 再读，结束后再读终点；原始行是 `(run_id, time_ns, counter_value)`，`time_ns` 相对该轮主机单调钟。单轮平均处理 PPS = `(C_end−C_start)/(t_end−t_start)`（计数下降整轮作废）。跨轮报均值、样本标准差与 `CV=SD/mean`。当前等间隔短窗口记 `run_kind=pilot`。图仅两张：图 1 为单绘图区 HAL|XDP 的 Mean PPS / 1 s ADEV（无三点则该轴 n/a）/ CV；图 2 为两行×三列——等间隔区间 PPS 轨迹、约 1 s 窗口计数直方图、噪声模型留出分（无 triad 则第三列标 n/a）。


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
from IPython.display import display
from utils.db import dsn_summary
from utils.experiment import (
    load_runs, load_counter_samples,
    mean_pps, summarize_variant, interval_rates,
    figure1_summary, figure2_noise,
)

plt.ioff()
plt.close("all")
print(dsn_summary())

runs = load_runs(run_kind="pilot")
assert len(runs), "formal.experiment_runs 无 pilot 行：先在 TR3000 跑 ./xpost-test，且 .env 设 PGSCHEMA=formal"
samples = load_counter_samples(runs["run_id"].tolist())
print(f"runs={len(runs)} samples={len(samples)}")

by_variant = defaultdict(list)
rate_ts = defaultdict(list)   # variant -> [(t_s, pps), ...]
win_counts = defaultdict(list)  # variant -> [ΔC in ~1s windows]

for _, row in runs.iterrows():
    sub = samples[samples["run_id"] == row["run_id"]].sort_values("time_ns")
    t_ns = sub["time_ns"].tolist()
    c = sub["counter_value"].tolist()
    by_variant[row["variant"]].append(
        mean_pps(t_ns, c, run_id=row["run_id"], variant=row["variant"], pair_id=int(row["pair_id"]))
    )
    dts, rates, flags = interval_rates(t_ns, c)
    for i, (dt, r, fl) in enumerate(zip(dts, rates, flags)):
        if fl != "ok":
            continue
        t_s = t_ns[i + 1] / 1e9
        rate_ts[row["variant"]].append((t_s, float(r)))
        if 0.5 <= dt <= 1.5:  # ~1 s windows for the hist column
            win_counts[row["variant"]].append(int(round(float(r) * dt)))

summaries = {v: summarize_variant(rs) for v, rs in by_variant.items()}
for v, s in summaries.items():
    print(f"{v}: valid={s['n_valid']}/{s['n_total']}  mean={s['mean_pps']:.0f}  "
          f"sd={s['sd_pps']:.0f}  CV={s['cv']:.4f}  per-run={[round(x) for x in s['per_run_pps']]}")

hal, xdp = summaries.get("hal", {}), summaries.get("xdp", {})
fig1, _ = figure1_summary(
    hal_mean_pps=hal.get("mean_pps", float("nan")),
    xdp_mean_pps=xdp.get("mean_pps", float("nan")),
    hal_pps_points=hal.get("per_run_pps"),
    xdp_pps_points=xdp.get("per_run_pps"),
    hal_adev_1s=None, xdp_adev_1s=None,
    hal_cv=hal.get("cv"), xdp_cv=xdp.get("cv"),
    title="Fig 1 — Mean PPS / ADEV@1s (n/a) / CV",
)
fig2 = figure2_noise(
    hal_rate_ts=rate_ts.get("hal"),
    xdp_rate_ts=rate_ts.get("xdp"),
    hal_counts=win_counts.get("hal"),
    xdp_counts=win_counts.get("xdp"),
    title="Fig 2 — interval PPS / ~1 s counts / model holdout",
)
display(fig1)
display(fig2)
plt.close(fig1)
plt.close(fig2)
